In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ranksums

In [ ]:
expts = {'XLI2':'Schisto',
         'XCE1':'CoV',
         'JYH3':'BoNT/A'}
expt_order = ['XLI2','JYH3','XCE1']
alg_order = ['individual','meta','scoper','mmseqs']

scores = []
for e in expt_order:
    for a in alg_order:
        sc = pd.read_csv(f'../fig3_meta-clustering_validation/{e}_{a}_scores_extra.csv',index_col=0)
        sc['expt'] = e
        sc['alg'] = a

        if a == 'scoper':
            sc['max_dist'] = 1 - sc['max_dist'] #fix mistake
            sc['scaled_ranksum_score_v3'] = sc['ranksum_score']/50
        elif a=='mmseqs':
            sc['scaled_ranksum_score_v3'] = sc['ranksum_score_v3']/(-50)
        elif a=='individual':
            sc['max_dist'] = sc['cut']
            sc['scaled_ranksum_score_v3'] = sc['ranksum_score']/50
        else:
            sc['scaled_ranksum_score_v3'] = sc['ranksum_score']/50
        
        scores.append(sc)

scores = pd.concat(scores,axis=0).reset_index(drop=True)
scores['max_dist'] = np.round(scores['max_dist'],2)
scores

In [ ]:
meta_method_order = ['single','average','complete']
mmseqs_method_order = ['connected','setcover']
score_order = ['silhouette','ari','ari_to_full']
shared_cols = ['max_dist','method','expt','size']+score_order

name_convert = {
    'ari':'ARI (true)',
    'silhouette':'Silhouette',
    'ari_to_full': 'ARI (full)'
}
cmap = plt.get_cmap('tab10')

method_order = {
    'individual':['single','average','complete'],
    'meta':['single','average','complete'],
    'scoper':['single','average','complete'],
    'mmseqs':['connected','setcover']
}

ylims = {
    'ari':[-0.05,1.05],
    'silhouette':[0,0.55],
    'ari_to_full':[-0.05,1.05],
}

for a,alg in enumerate(alg_order):
    for m,method in enumerate(method_order[alg]):
        plt.figure(figsize=[13,10])
        for k,kind in enumerate(score_order):
            for e,expt in enumerate(expt_order):
                ax = plt.subplot(3,4,k+(4*e)+1)
    
                subset = scores.loc[(scores['alg']==alg)&(scores['method']==method)&(scores['expt']==expt),:]
                if alg=='individual':
                    subset = subset[subset['merge']==0.25]
                
                sns.lineplot(data=subset,x='size',y=kind,marker='o',hue='max_dist',legend='full')
                plt.semilogx()
    
                if (k==0) and (e==2):
                    plt.ylabel('Score',loc='bottom',fontsize=16)
                    ax.annotate("", xytext=(-0.22, 0.3), xy=(-0.22, 3.4), xycoords='axes fraction',
                        arrowprops={'width':1.7,'color':'k'})
                    
                    plt.xlabel('Downsampled Size',loc='left',fontsize=16)
                    ax.annotate("", xytext=(1, -0.2), xy=(3.4, -0.2), xycoords='axes fraction',
                        arrowprops={'width':1.7,'color':'k'})
                else:
                    plt.xlabel('')
                    plt.ylabel('')
    
                if e==0:
                    plt.title(name_convert[kind],fontsize=20,y=1.05)
    
                if alg in {'mmseqs','scoper'}:
                    legend_title = 'Max. Dist.'
                elif alg == 'meta':
                    legend_title = '$d_{meta}$'
                else:
                    legend_title = f'$d_{method[0]}$'
    
                if (k==2) and (e==1):
                    plt.legend(loc=[1.4,0],title=legend_title)
                else:
                    plt.legend([],frameon=False)
    
                if k==2:
                    ax.text(1.05, 0.5 - 0.037*len(expts[expt]), expts[expt], fontsize=20, rotation=-90, 
                            transform=ax.transAxes, color=cmap(e+3), fontweight='bold')
    
                plt.ylim(ylims[kind])
        plt.savefig(f'figS{a*3+m+2}_{alg}_{method}.png',dpi=300,bbox_inches='tight')